# NB02 — Execution, Independent Risk, Governance, and Audit

## Introduction

This notebook turns a promising research result into a controlled institutional decision process. In NB01, we compared predictive models, translated forecasts into stock-level signals, formed constrained portfolios, and identified a candidate strategy using out-of-sample evidence. That candidate is useful, but it is not yet trustworthy enough to trade. A backtest assumes that desired positions can be obtained at observed prices, often hides implementation frictions, and may compress risk into a small set of performance ratios. NB02 therefore changes the central question. We no longer ask only, “Did the model predict well?” or “Did the portfolio earn an attractive simulated return?” We ask, “Could this proposal pass through execution, independent risk, governance, and audit controls without acquiring authority it has not earned?” The distinction is fundamental to the course: predictive intelligence can propose an action, but an autonomous investment institution must also control, challenge, document, and, when necessary, refuse that action.

The notebook begins from the governed handoff produced by Day 4. Its principal inputs are the selected position history, the selected portfolio’s daily returns, the synthetic daily equity market, and any supplied corporate-action records. These files represent different layers of evidence. Positions describe what the strategy wanted to own; portfolio returns summarize the research simulation; market records supply prices, volume, spreads, highs, and lows needed to model trading; corporate actions address accounting events such as dividends and splits. The inputs are embedded so the laboratory remains reproducible in Colab, yet their role is unchanged: they are inherited evidence, not newly generated truth. Cryptographic hashes bind the Day 4 position and return files to the Day 5 review, preventing a later substitution from silently changing what was evaluated.

The first analytical layer is execution simulation. The notebook converts changes in target weights into explicit orders on a one-million-dollar reference portfolio. Each order has an identifier, date, instrument, side, quantity, type, and authority label. Market, limit, stop, and scheduled-order behavior is demonstrated, while fills record status, quantity, price, spread cost, slippage, market impact, latency, and reason. Liquidity is approximated from participation in average dollar volume, and a square-root-style impact proxy penalizes larger trades. The simulator can reject orders that exceed participation limits, partially fill larger requests, or alter costs under wide-spread, latency, and liquidity-crisis scenarios. These are simplified research mechanics rather than claims about a broker or exchange. Their purpose is to expose the implementation assumptions that a frictionless backtest leaves invisible.

The second layer is reconciliation. Orders and fills are joined into a ledger and tested for legal states, duplicate identifiers, and impossible combinations such as a rejected order with a nonzero fill. Reconciliation asks whether the execution record is internally coherent before any performance or risk conclusion is accepted. Corporate-action accounting is also handled explicitly: adjusted prices remain the return basis, dividend cash is reported separately, and split events are counted so that the notebook does not double-count economic effects. Futures rolls and derivative margin are marked not applicable because the approved scope is synthetic equities. This is an important governance habit: an unavailable test is not quietly presented as a passing test, and a one-asset-class experiment is not allowed to imply cross-asset validation.

The third layer is independent risk. It is intentionally separated from strategy selection so the proposing logic does not grade itself. Using realized portfolio returns and position weights, the risk module estimates annualized volatility, historical value at risk, expected shortfall, maximum drawdown, gross and net exposure, maximum single-name exposure, maximum sector exposure, and the minimum number of active names. A limit engine compares selected measures with declared thresholds and assigns PASS, WARNING, or BREACH states, together with a human-review escalation when a threshold is approached or exceeded. The calculations are transparent, but their limitations are equally explicit. Historical VaR is not a worst-case loss bound, synthetic returns omit unknown real-world mechanisms, and sector stress inside one equity universe cannot establish resilience to genuine cross-asset contagion.

The fourth layer is stress testing and deliberate failure. Instead of treating the base case as sufficient, the notebook perturbs execution conditions, market paths, liquidity costs, correlation structure, and parameter effectiveness. An equity gap-down path, doubled volatility, an equity-sector correlation breakdown proxy, fivefold liquidity cost, and a weakened parameter response challenge the candidate from different directions. Separate negative tests submit an incomplete request, attempt a live order, force a gross-exposure breach, simulate a provenance mismatch, and request an unauthorized override. A well-governed system is expected to fail closed: missing semantics are rejected, live execution is denied, breaches escalate, tampering is refused, and material overrides require human approval. These failures are evidence that the controls operate, not annoyances to be removed.

The fifth layer is governance and audit. A semantic-sufficiency gate requires an action, artifact identifier, scope, and evidence. An action-rule gate allows bounded simulation but denies broker-facing or production-changing actions. Material decisions, including strategy promotion or a limit override, cannot be self-approved. Every significant step produces a hash-bearing audit event with an actor, action, timestamp, input hash, output hash, and governing policy. The final bundle records responsibility clearly: strategy proposes, execution simulates, risk independently challenges, governance enforces, audit preserves evidence, and a human remains accountable for material approval. The notebook consequently distinguishes a successful control test from a successful investment claim.

Within the course progression, NB02 completes the transition from isolated quantitative research to governed capability. NB00A established the operating protocol and authority boundaries. NB00 built the synthetic market environment. NB00B demonstrated the smallest end-to-end KNN prediction slice. NB01 expanded that slice into comparative model, signal, strategy, and portfolio research. NB02 now surrounds the chosen proposal with the control plane required by an institution. This closes the course’s first act of building evidence and prepares the next act: converting reliable functions into registered tools, repeatable skills, and bounded agents. Day 6 will package the validated schemas and controls into capability registries; Day 7 will coordinate agents in constellations; Day 8 will introduce a governed meta-agent; Days 9 and 10 will integrate the autonomous research system and its research-to-LEAN bridge.

That disciplined separation between evidence, decision, and authority is the durable result of NB02 and the foundation on which the rest of the autonomous investment institution will be built.


## Workflow: from a Day 4 candidate to a governed Day 5 decision

NB02 is designed as a sequence of gates rather than a single backtest. Each stage consumes a defined artifact, applies a bounded transformation, and produces evidence that the next stage can inspect. The workflow is deliberately fail-closed: if semantics, permissions, reconciliation, provenance, or risk limits fail, the notebook does not silently continue toward promotion.

| Stage | What enters | What this notebook does | Evidence produced | Authority after the stage |
|---|---|---|---|---|
| 1. Reproducible handoff | Day 4 positions, portfolio returns, synthetic prices, corporate actions | Reconstructs the exact research inputs and hashes the critical files | Local input files and SHA-256 provenance | Read and analyze only |
| 2. Semantic and action gates | A structured execution request | Checks required meaning, declared scope, supporting evidence, and requested action | Gate decisions with explicit reasons | Simulation may proceed; live actions remain denied |
| 3. Order construction | Target weights and market closes | Converts weight changes into dated buy and sell quantities for a $1 million reference portfolio | Typed `OrderSpec` records | Simulated orders only |
| 4. Execution simulation | Orders, prices, volume, spreads, high/low ranges | Models spread, slippage, impact, latency, partial fills, rejections, and selected order types | Typed `FillSpec` records and an execution ledger | No broker connection or order transmission |
| 5. Reconciliation and accounting | Order and fill ledger, corporate actions | Tests legal states, duplicates, rejected fills, dividends, and splits | Reconciliation verdict and accounting summary | Continue only if records are coherent |
| 6. Independent risk | Positions and portfolio returns | Measures volatility, VaR, expected shortfall, drawdown, leverage, and concentration | Risk state and limit checks | Risk can challenge; it cannot promote |
| 7. Stress and negative tests | Portfolio path, exposures, executions, permissions | Applies market, liquidity, parameter, semantic, provenance, and authorization shocks | Scenario results and deliberate-failure decisions | Breaches escalate or fail closed |
| 8. Governance and audit | All prior evidence | Assigns responsibility, records policy decisions, and hashes the event trail | Audit bundle, event log, reports, and decision | Human approval remains required for material promotion |

The executable cells follow this same order. Run the notebook from top to bottom in a fresh Colab runtime. The dependency cell establishes the small scientific Python environment. The embedded payload cell reconstructs the Day 4 handoff and synthetic market so the exercise is portable. The engine cell defines immutable order and fill schemas, execution and risk functions, gates, stresses, provenance, and audit events. The orchestration cell invokes `run_day5`, writes the Day 5 artifacts, and exposes the decision. The final inspection cells then separate execution integrity, base risk, stress evidence, and deliberate failures so the reader can evaluate each claim independently.

The output directory is the notebook’s formal handoff. `execution_ledger.csv` supports transaction-cost and reconciliation review. `independent_risk_report.csv` and `risk_limits.csv` support independent challenge. `immutable_audit_events.jsonl` provides a hash-bearing event trail. `day5_audit_bundle.json` unifies decisions, limitations, permissions, tests, provenance, and responsibility. Day 6 can register these bounded functions as governed tools and compose them into skills without granting additional authority.


## 1. Establish the reproducible execution environment

The notebook uses only NumPy, pandas, and Matplotlib. The installation cell is intentionally explicit so a fresh Colab runtime can reproduce the laboratory without relying on hidden session state. It creates no external connection and grants no live-market or broker authority.


In [ ]:
!pip -q install pandas numpy matplotlib

## 2. Reconstruct the governed Day 4 handoff

This cell decodes the synthetic market, selected positions, selected portfolio returns, and corporate-action file embedded in the notebook. The two selected Day 4 artifacts are moved into `day4_outputs/` to preserve the expected stage boundary. Embedding the payload makes the exercise portable; hashing the critical files later makes the handoff auditable. The data remain synthetic and the candidate remains an untrusted research proposal.


In [ ]:
# @title
import base64, gzip, pathlib
# Compressed, deterministic payload: self-contained in Colab without a multi-megabyte raw Base64 cell.
COMPRESSED_PAYLOAD = {
    'prices_daily.csv': (
        '...'
    ),
    'corporate_actions.csv': (
        'H4sIAAAAAAACA3WW4U4jMQyE//MsKUoc20meBlW00lU6CoLlpHv78+JZnRD2L4R2Gk/Gzpecn7fb6/3pdim3+8f2/vlyvW/7f5fzdi1n/7j9fbuW5/PHr6fzy+vnfSsfb79v29P72b4+tNIKVeqnuk4k5XL7c7tc75dSH6nqKO2xPnChRNImrS/JKD2VcPuStFo40egUcU0vkmm6kGu0aFZrze6aVUaiEXE7RGVmpaq6RMrKKlVyxzSLbSwWjT49nW4ls5SVl6/UubQs596Gb75bR7KkebKX41paFnWf6hGx/ZHUeJsusiCysFWWJ7lXTdOGpb13adzkY7bPwEptN09J9lXS1vKXRi3rLG5eDJEllMWtg3wC7ABQGveqbmmYnyzuRt03N2yZLG6p3bs7LOt0tnm6p2EZZXEfYzJtZ1nc1DBw0/xkeTP7AMxZek3PwMA5cYTwLqnrW9vqRG8dIYFEl/oYTUdIJKHubuwQcabp4pVs8iVzo4p1hiMk0NBEV23KRqKROnyqbTZmts5SrKMOkcizsnveIVoT0eDlpvcZSXP2RnQBQgIJN5zpPcss6Xa43pmdRS3a/QjtJynNmgc4M4CQSLSqe5IKhERJyvQBkQ6GRFHaQXORAiJhObBfFiASNg5TrQSKRGFO8XIWF6WJH3eETR3liTcc6waKRJ4UqB0MikS7m8dKAxSJyjVwdFZQJGrwwnmz25TSAW/VIzAsUZa4zOUTbhdhzxPHZNomsblGp0r+TDGB/XD/SmianKqcrMnfP3eHkH2cP+4OXELqEAokpNUlyyEUSQT7tRPGmWYOvHXEdxJp6vGOmQ6hQKPHyNsAjmxXY4G+7BAKNE3BqeEMCiRV8PSy9rSaeaYDMB0MikxPnK5dmwUtCtd9gULR1vAgYAKEIt8DUbMAQtFCvQNCExAKRKy463b7Wdqj49m0PzHSuNvxkhmAUGR84fLQ+n+efyYggFAHhKLAGfRUBYQikQXlogUIRZ6UwBcChCJPjEeKgEFRtUVuaUwwKMqyCvDSwKComuCdNhkMisoNPJzslUFZ4Nybr7QqGBSJSLwr/wBxsGt/UA0AAA=='
    ),
    'selected_positions.csv': (
        '...'
    ),
    'selected_portfolio_daily.csv': (
        'H4sIAAAAAAACA62b2XIkR3JF3/UtaFjsy9fQKLFtZh5EmkjOUPp7nRORWYlCo9oaUhnZTTBQFeHp4X79+pK//Pzn15dfv/750+9f//zn77+++Ndv//r6+8vffv/tjz9e/vPn//7p53//46e/vv7jb3//8+U/fv7j7+fPX//rn//4839efvn9579++e2vX/8thVS/BP6NL+E13P2Jxx9+vj6WXr7w/yH0WlsuodXR5syp+akSWeulpFZSLWN8Ym3WUXJLtfZce6kKM+dMhcU6c4n5o2NHTJdg9eVLeX238VeXHwhxe8R9Ukgj5zrbKG2cZ80eWyshx5rjzPk6q32oq//DRv1ZG52fiDxl6SHm0nLNoWU3zX00ftFbZTW28om12VLMI9WUS81t5C0botVUW2ltxHKcjGilFw7NcdQ2xiXbdO8QZ61h8kA18kDdJy8xhTC4k9Tj6NlL55b89SwIkkpcio6zzThLHImfUIJfxVQGOggV4Tq2o96QYcQe50QA9ryz27j+N4TaY8qp9Fln7Oy6pIg8MHskvpfmobEaOmK0ENPo49Ea2ikzIUaeg43TkiLWmmtEC1h0L/dS5C3FTLHgNbVk7nmbYwi5IRaqbpnVNts+M5Y0JtdZZuF2Hqwhx+DJ0CU6rikuOXjKhjHEUFJu816O89JyKKGPycWgbnyPL9YwEnaTSpoh11ZPH+LY1Aeio+NQZm2PFhWmc1WBS02zblm4lxZT5JpnqR8dHmdql3x8JL++3/u7krxFLI5DOSh0cDfphA5QBBvEOYq/bZeBxvYI+j670XzSRik8a6P4rI3SszbKz9qo7U80/KZ2QKmAF2U5NBYCHOEKvY8Ya3mwBooNvhwjbj1xnawjArgYPoFnJvBg1AV4tdeW8gRuwiwH1LaOqCEPPH/M3volWN941zMeIFbUKpJrZGXg890IVoC7dR6YPsDdgCR4QtqBNIthk0cefDou1CAeRLB4xobbzIWBc4LVwBHhUmjuW64yB8JXQTykPC6nSmeM4NEIn4NAAqqW6s0mPtRC0+EB8bKEjSWIzOD1rKDBsiSALVSUhdCDn/zqANF7T6GMgLB9LsH4CEiGsOh87mN58NJSz0QmPt17vQQ7AgQq69zEBPD50AK22EFkgv3glhV5h6lCsOh8hviQ2gFE367xvZgaWFYryskrms2Bmc0eRg9Er4PLYFY8X0pYUABNrpvM2mXlfjWawKOHOk8QagSk3CJ3PCAUYUcIYiQhaC5hiYYP1qbxNUS+1DSCegiGtYGDmdDX8ykYETZWzWhmuc8hWZOG5dcReLoKtUHLmMwl2SAiLnZEfM1bMsJyC5yBPtXmg7VpSMfOCUPcVoyHZIRzDH3IOk7B0CAaM/7rAJdcWeB+t/ENtz8Q4o7zDCAAloECZoz1poOMP9ZmvMDNr6PKYTVreZE7Lm8zqsyVGxlihgLVOeqnFtECnlGx6NbkngfnwTshYxHNjLhl43cQgoyyM+6LF1zCrQD2fu9LER9Ics/+AqZSeTi8qPTzNAgPIDdgZkBdGtdp3+Gjn9toPmmjGE6swZPh9gAvpIh/+wFvOSVQL8kZN0f6wTUJYAMlIBRy0M1Hh+pErthAvcNuspiRO1heFiNrl2zRu3m39e1qPhDj3kaBVZByiuEHoGaoS4wlZ1EcznWdlB6q81Pb5Ods056zTX/ONuM528ynbPOAcH16m+fcVNrQBrUhasaKCcOa594cmIOV4HgEI9xpY8cPLxKqM0cSf+BLOO12bWh0gqKUSl5yxMScpj6VCeJwlzLzJdxKtN/t3W+Z9keS3AMJ5B0+RfTIl7OS0kO8wBZiZQF5r9PaVgXReZjQwuMS6dqGeTLJxbEquJ85+BNr0Dh4DGmU4cWAu0RD4kSUh32C54dokVAE1DT+AmHexGB43pf6+m7rUw8fiXGvBu6WYJJlSLmcZ0GQrHr0HmFiF2Tlx+b5iX36oyLP5/dJmxgRvJrUChXeHjyjzcq6rORQ/48sYY7oSf5P+ovKDqG47A6ta+b2p1AYL+APK4RNQtQvqbIXcrdzn9+R6129BZJjPsq93Z4/EcBgqqih+vd1UvlO3eYz+xyBNFQLEovIw7rGziBwESw0yuQmlKN9Ym1ymsxUW4ZTjY1AeOboBbOv+czBM8SWtIuPofQ6yhvRhsp8t/VNmR+IcY920F30TRJgUemMyJmMi8QHGuwtjOus74D4pzaK4VkbxQN2IOSYOQnQKorFjXFNfuzn5e87J/nBNVQv/cXETdnakZNMPgHHTYDgybCliNNkGIpo3ndZubWb8nq/db7B7wdi3KvBCh+mhnHOm3k27A+ggnyDAiVfR9XH6vzUPu1J+/Qn7TOes0+KT9onPWmf/KR9lt1EML8EE33+O9pahBFEefE0YYaSfLwE5ABnkHQjbJqrmg/sQQwSmT95UgVdkAxsIqcC8sgS5qxvSpR90YydQfzYkXeVJGvReJVPfOZq9/vkNw87HhakPrXNfMo2OTxnm41fkEogv0aYCL8papjcPloyS+RyjbDeH6xhJwRkk6xFz1ZIskYB2NReIGNE4HWHvcBfu7EbHG1v7nBINxQzJuJ1sHpvpQ1uuK4wTgG4WAepC7wsDJFJw+281XhUwiBfVrwam+QYFpK0Ci0IzTIWpjT60k6BOnaNrfihD44d5ZLrLH+nTAxbdW4idslLCjjrSE0GCIE9ksIIBYFsBEWYuT1Yg1YkklOsPjQu6pALGu//d+6uHoLJByAiEt5oeesSrB7Ev2ez5VY7OUNZxT4+GGFH5NK2KcIu6EPVLNd7Tb3ERZogGxDohA4L38hH1hy4Hmh+J6tNN40hLiJk8vZSx6EyD5W6dxRaerokO3g4G0rFraUWsGMLZk8NTsCtwB93vSlakYIwYzuQ+vRgDY0F+ykYVAOftlw8XLQMZ+yrh1iY57Q+RGpBstEusfrLfB1S82FxLllBPWPhzB1rthuBSZQFaS2icPCsDzscYXk+N4gtrRuD7y5mN/OIo8dhvRXaOLdYFjPN05KFh0OuAG3oQe02PGJech09HU3JVGjgohYwl1wYXLCaa3znp62bMCAnnRvDjI8e1Ldr7EH6hC8Q31s4+jy4Y2vgKm6BdFuwWDR6bithR5CXm2AWPyJ31gq8nUW4DXZ/Kx3a+UHVPD9K2uEf/fDMNu3glPMsUH+wOOUwGTvHPVPbCB/s52KHSEsqe8jmJWFiYXfwxiXb2aODCC2ca9rhDkZoPe5KqP3BHcDtEtRkQ9j7m/XBGkrrR1U7E+i20tpycdGIjG9zLtx4GIqQHlMjHbkEO5sRCaxD9AL28bS9He08EJhEuXICTi4s3K7Psu1M0vv+cHFfarDwTZq/e4j6BNCR+DOPMnWMKZmICgkAcXujt6NcEFftXB+oNji50v7aTaaKCrcofEtPhVKuYBIbho2R9mhR4ezGhlXlS9sVmm08Ej9kCeW8VZ7Hiy7VSmF8o7wzkK90QXsG8XjaTXpwdYxXamARb1e6MK66NgMHzaLao0UtrhCaAmGDWLANbgoqBbdr/US1CCSCMTg1l4baL9kON51lcW1SAUv0M274sMUD3k6O7ad/RBwDlx9opdxcF3Mb+Kg9o5VhgzxjlR/5uNFo6cw+Cg8ovh4XyjM1G0HaTenxQrU4X7601/uz4uWj38pwxxNQJlF0AKco9zRtYcy+fPYm42U7u/DlNZIsI4R9itm3sdoQSpI4zIrHy4/WuA0eLC4Ohw0uEjD5HdGJ+LhC3RKMJzXXHByEbrZVkC8Dh83+TfZ2LsE2g4mGM40x1eYIx444wWIVcZBLxeLiFmwFcy85r+7FlgzQIBnGi2ABaREb5Wo2r/g6MWeLhkzAXhpYwDnhAmGOlcgzRcMWL8EOlHIEI3U7k8Js26qYzkBoNxAUKPBhIHbPui22bg/z4aJAVa0891XfOCQjRx1lBfB2eBLheoLt2X46X7lcLR30AatDmXiVQDDKRlAseRhuIF62pJbOCiDJYiNOFvjApg+EhcLDWZMMu8AuE8K17XKkOg+5gCprZatTefApKBOSBBui8Lf65jKbBv3utJtBfyDFnUEPM3Cvr+Gh5+2I1HhUMWlG59dR/aBQpsGjcsu2EhcfXtQHR8cv2IjvPljbJaahv3JHcGwXJVhJipHtg40dRQBZO58AGt51FAy7tdoMaMAXsKDLq2/9V0Eh2EO16pJ2FdZ2LycQ4ICaMXfmjtjTSi1UEN88I0i1QIePevW7awdyGU9XI9OeyBIt2xnGE0SwQ2ndlvJccRRoeBN5jw6sqMzzQsVsuM0Vj6NJxijyprqo2Z7ASGJctImNRc0Ha4SuLl5BgvTtuHM+0LbVinfYkz8EwxVAd4KeJb103uZcGUR+fbfzNQfyrRRvDcd+rfcoWTVWH0c562Mo08VDvY56OOHwyX3yk/YpT9rnAAXIoT3cacd42nfY9SKpdnb4ysbADy/NStIHoYqCTJtbqqw4FtBrPFhBcHBL766Lvlwd7Lnqi+X1fuOrivWNCPfP7xgC5lmcQzsPwqZwYSg83BcOeh00H+vxM/vE8KR94pP2SU/a5zaOBYTh+NYjlxUtICXllCgD/QuvNsn+4UWQCw9lV+wtESo2MrEKaSPdL6eZ9D0KZ8bn6EK7hGtngbw6HjTiolo9nyBvrlBtJAGPpPhLDqhBxwBjBUgg7A/WMDB5fK3DyvxBAwBNU2rn0aS/J2jibs3sTZ74RrauBb/b+WbCH0hxX82xwkuAdWxl3I4qQrn342ThBYPw1C/pFTkd6yR3Jt0lxlw9N1JEG01rtnKPnfzgGmHKAUyUT+SFHh6ipe44IVFNHnWI1qPzWYZK/nmjhakWxBQuejTYPxd8Ew1CQwyaZtLS3H0ZPAP7EmGcDxz9wdpcBRBYBDkXQfEIuMlqR7Me2U7jMRlyzMoOipOON9Fgaoj2bueb0j6Q4v6CoIuctZ543I5qBprhwBnsJ11HbSLs80fJMSnaXMlFsaeZRli2A53sH6wtcaRYTsPO6bxLXpNosFMYeXT0xKFlpHLKQctqVoLym2LbPIu2kIZ1xaGYea/iJsqymJVXJWGMNe8FTrMcnO2FJa2dInmOpbg1qoRV+jmiMdlgnlaPZUXKYOBOgnl0TO2dECdFRW9Jamg1ae8P3cYCyGFFibpd2Nt1LpSwj/duUmqai6SWtcg9u2jpNZk52o0BxJYpVHMp8pue1lzpB8fOeDnQ2UbmcYE2ng4RZh9bhoaZweLNxc4Oj1pJsWnOgNLY04JOICJXtqINi16UvaqXbos8qLctmW5aTZM1kpMad+efHNSyzDouyeY5241zYRBC9TwKnabyK7mwMbjIvQN9juB1O1OoYJktGU/t3S4WOW7Yk8k8icWIYg3Ucu0ynlWSD2vaOp/IOmXwcU0wG8ovwfI5wmObFeHxR9uHKy8j57VgNsBjEtqdV2EPxZpotEAVd9PRAUTspVtN5EF2vogPVwulQ/5KrrDt2m6Xn0yy6zM3hBVbInIsEfm2aDEcbewgmSXJhy9rLM5KbDFYmn1nO+y47rObnCOvt4Tx1S0buinOjUrx0/EaQjLUrLTRNGCrzfp3da4e8U7ZanBYdZF9n/eS7UwPczZd2sWFfkQA4RYHHdYvEGaTq77eaZgOR0ZnNw+9Oac6pbZ17lrJNJQUmzaYxNiA6AS4c+uL84Uzp85dc8Esyac45ZLtHD932J24l4x+dZeXi2ZsUmxxMm3DmnVX4o3WiJQPydZMA0Az/P660VB2Jb6aCO8yrpDls87h7Z1JG6ICplqwBb5LsO2e6DNbQCg2FeYCquhcuAUyax4st2ON+54ZmbnLfojlBY2V69VFbodjm72FVk0by0qLnL41ybefFY+rLN5HtIZN3OUWLqkOlsEHMIthNc6BzK0uFANsYFNJVz9UszBComU3IT1Ywws1zzUINI+yVeVupTfiyjjLjdpEsVg9te9zPF7BDskj8q4+hAOzYxwKqxmNNTXmUOuCXpsZwTlO31do9TD/1fCXnRVL22uWlyy4+K6FPMzxhbgGJ3xxAzqQhbpzmDc5cOuUb3P+8hLtgLPIlk60GqnI3Fe5zqlfYi6UB5XqHntaFeGItQ47Hy+YWMlJEI7ih60lrBzX+oCD2iY5eVVJ4X6E0Szw8Wwn0XXoyOoUquHHSzJHEtvr/Wnpmpn9Voq3bACOadcNEEiWxo+jGtaEa5BkO2tRrqMepoCf3KecUcsI7oQ4ALJrteslFucC4HLaffnUovPq1SnHCu1z5mCJ1i3Dqgrc5hxyB9v0/hw1jVwv0Y6sMlkb1tq45n70cG1wdKd4R+u+q7IVbAsA8kLcwjwPC0ziI47Lg4+Nv9mqIlcPP9FYtgEuDr5Gu+eZrfhelHFbTOe5LrnOeTGrJNn6mJXNnI9s1xzIoi6s6ozqGCKBnmDqxO+p3G/W0FizyEssXZO7WzBbDRAK/pvPGpiv4Vh2MarAAvMlWb81HQE6Z++RxTbFIVqJ1i4dnQNpdnlu4q3FWqVx+/BZq0/O6Q+nSLbGDIGrkenrRksuy0tJF/aViFNheQ3/QQjW2wE3uY4KMBcYnJWtPk2vB5RU51/sRsxDSpsjUbGyN35o0JwSWld6NL5t8p65e+m/w+ObFpU6ILvrZQFM8cBeJxEl1Ogxzjch/ij/4vRdVC5OIfTjVak2TdTGqr62uW/czDbbvXLguB5BFBMyTK0XPuxVLXVhEs2JMrtLpR7ga9o1BNl0zkZMX2Mj0viuG0B5CbbjezL6ex8GYN+A2M2M1ZxxYNDp4VWrdrZq2p6dotTRoTU4Ggr1vgVA621Gu/SgriFgiaX0yTYC3lUP67H6B8ah6LEymAvfbm/MQKSBdu6REJlsbStGs6xY+Kv58toWQ1IXLbwSmzj/wZrFAB/KUXiy+y2ZelkvqEhQ9rmgOxkYbhy0z3AF92OaNdriKbuontrmOVA9s0Ljw0prx6rI+uIJf8zK4u5zWxYmLyZYrRLDmkUQ5qss3txvd/UsUuOI3Mqq0B7ullbne64G00xvbvI2fFfWqy6SeKBvtS+iTQnfr9vNvV3cBx19NcAkJFmB3WkkYjrKSq4m/VZhBF7nt8xpSFWXYHzT4ql+X082FHya9VaDbGfGNy4J+o7X+9Nu5fKPpLgLMcNWjrVr7ffk8NW4O/xS8f2f66jj3ZskEUjr7vHHs6JTSBfsZxqP2zHU9qOL5AvV3nBoVqp3Vdph4mFyRWCZ25lvkphvkK3fbfQmW//g1Lunnma3c/f2w7zfyqx69bivwx4W4T61UfxOtfezG5VnbVSftVF71ka3yFeNE1p+cJbjsDIHIUV/X1zrO7X7wTWLxKS55i2247ZosELox3qF8Cx+AiXW2qYQPyweXbLFZXXvtr6KeN+Kca8HMw5beJx2vbGYJVq+hxucti3XWd8xus9tlJ61UX7WRuVZG/VnbTSetdF80kaPX+r97Eb/Xzv6X+Dz+bI+QgAA'
    ),
}
for n, v in COMPRESSED_PAYLOAD.items():
    pathlib.Path(n).write_bytes(gzip.decompress(base64.b64decode(v)))
pathlib.Path('day4_outputs').mkdir(exist_ok=True)
for n in ['selected_positions.csv', 'selected_portfolio_daily.csv']:
    pathlib.Path(n).replace(pathlib.Path('day4_outputs') / n)


## 3. Define execution, risk, governance, and audit controls

The following engine is the notebook’s control plane. It defines typed order and fill records; converts target-weight changes into orders; models spread, slippage, nonlinear impact, latency, partial fills, and rejections; reconciles order states; handles scoped corporate actions; measures independent risk; evaluates limits and stresses; enforces semantic and action gates; verifies provenance; and writes hash-bearing audit events.

Read the functions as separate institutional responsibilities. Strategy data propose positions. The execution simulator translates and costs them. The risk module challenges exposures and losses without changing the strategy. Governance controls permission and escalation. Audit preserves evidence. None of these components has live authority.


In [ ]:
from __future__ import annotations
import hashlib,json
from dataclasses import dataclass,asdict
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
import pandas as pd

@dataclass(frozen=True)
class OrderSpec:
 order_id:str;date:str;instrument_id:int;side:str;quantity:float;order_type:str='MARKET';limit_price:float|None=None;stop_price:float|None=None;authority:str='SIMULATE_ONLY'
@dataclass(frozen=True)
class FillSpec:
 order_id:str;status:str;filled_quantity:float;fill_price:float|None;spread_cost:float;slippage_cost:float;impact_cost:float;latency_bars:int;reason:str

def sha256_obj(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()

def semantic_sufficiency_gate(request):
 required={'action','artifact_id','scope','evidence'};missing=sorted(required-set(request));return {'allowed':not missing,'missing':missing,'reason':'complete' if not missing else 'missing_semantics'}

def action_rule_gate(request):
 if request.get('action') in {'LIVE_EXECUTION','SEND_ORDER','ALTER_PRODUCTION'}:return {'decision':'DENY','reason':'research_system_has_no_live_authority'}
 if request.get('action') in {'OVERRIDE_LIMIT','PROMOTE_STRATEGY'}:return {'decision':'REQUIRE_HUMAN_APPROVAL','reason':'material_decision'}
 return {'decision':'ALLOW_SIMULATION','reason':'bounded_research_action'}

def build_orders(positions,capital=1_000_000):
 d=positions.sort_values(['instrument_id','date']).copy();d['previous_weight']=d.groupby('instrument_id').weight.shift().fillna(0);d['delta_weight']=d.weight-d.previous_weight;d['quantity']=d.delta_weight*capital/d.close
 orders=[]
 for i,x in d[d.quantity.abs()>1e-9].iterrows():orders.append(OrderSpec(f'O{i:08d}',str(pd.Timestamp(x.date).date()),int(x.instrument_id),'BUY' if x.quantity>0 else 'SELL',float(abs(x.quantity))))
 return d,orders

def simulate_order(order,row,rng,scenario='base'):
 qty=order.quantity;adv=max(float(row.close*row.volume),1);participation=qty*row.close/adv;spread=float(row.spread_bps)/10000
 shock={'base':1,'wide_spread':3,'latency':1,'liquidity_crisis':2}.get(scenario,1);spread*=shock
 reject=participation>.10 or (scenario=='liquidity_crisis' and participation>.03);partial=participation>.02
 frac=.5 if partial else 1.0;latency=3 if scenario=='latency' else int(rng.integers(0,2));slip=(2+20*participation)*shock/10000;impact=.10*np.sqrt(max(participation,0))*shock/100
 if order.order_type=='LIMIT' and ((order.side=='BUY' and (order.limit_price or 0)<row.low) or (order.side=='SELL' and (order.limit_price or np.inf)>row.high)):return FillSpec(order.order_id,'REJECTED',0,None,0,0,0,latency,'limit_not_touched')
 if order.order_type=='STOP' and ((order.side=='BUY' and row.high<(order.stop_price or np.inf)) or (order.side=='SELL' and row.low>(order.stop_price or 0))):return FillSpec(order.order_id,'REJECTED',0,None,0,0,0,latency,'stop_not_triggered')
 if reject:return FillSpec(order.order_id,'REJECTED',0,None,0,0,0,latency,'liquidity_limit')
 side=1 if order.side=='BUY' else -1;px=float(row.close)*(1+side*(spread/2+slip+impact));fq=qty*frac;notional=fq*float(row.close)
 return FillSpec(order.order_id,'PARTIAL' if partial else 'FILLED',fq,px,notional*spread/2,notional*slip,notional*impact,latency,'simulated')

def execution_simulator(positions,prices,scenario='base',seed=42):
 merged=positions.merge(prices[['date','instrument_id','close','volume','spread_bps']],on=['date','instrument_id'],how='left');detail,orders=build_orders(merged);lookup=merged.set_index(['date','instrument_id']);rng=np.random.default_rng(seed);fills=[]
 for o in orders:fills.append(simulate_order(o,lookup.loc[(pd.Timestamp(o.date),o.instrument_id)],rng,scenario))
 od=pd.DataFrame([asdict(x) for x in orders]);fd=pd.DataFrame([asdict(x) for x in fills]);x=od.merge(fd,on='order_id');x['total_cost']=x.spread_cost+x.slippage_cost+x.impact_cost
 return x

def reconcile(executions):
 legal={'FILLED','PARTIAL','REJECTED'};issues=[]
 if not set(executions.status).issubset(legal):issues.append('invalid_state')
 if executions.order_id.duplicated().any():issues.append('duplicate_order')
 if ((executions.status=='REJECTED')&(executions.filled_quantity!=0)).any():issues.append('rejected_with_fill')
 return {'status':'PASS' if not issues else 'FAIL','issues':issues,'orders':len(executions),'rejected':int((executions.status=='REJECTED').sum()),'partial':int((executions.status=='PARTIAL').sum())}

def corporate_action_accounting(positions,actions,capital=1_000_000):
 if actions is None or actions.empty:return {'events':0,'dividend_cash':0.0,'split_events':0,'note':'none in supplied window'}
 p=positions[['date','instrument_id','weight']].copy();p['date']=pd.to_datetime(p.date);a=actions.copy();a['date']=pd.to_datetime(a.date);x=a.merge(p,on=['date','instrument_id'],how='left').fillna({'weight':0})
 return {'events':int(len(x)),'dividend_cash':float(((x.cash_amount*x.weight*capital).where(x.action_type.eq('dividend'),0)).sum()),'split_events':int(x.action_type.eq('split').sum()),'note':'prices are adjusted; dividend cash is reported separately and not double-counted in returns'}

def independent_risk(positions,portfolio_daily,capital=1_000_000):
 r=portfolio_daily.net_return.dropna();loss=-r;var95=float(loss.quantile(.95));es95=float(loss[loss>=var95].mean());equity=(1+r).cumprod();dd=equity/equity.cummax()-1
 bydate=positions.groupby('date').agg(gross=('weight',lambda x:x.abs().sum()),net=('weight','sum'),max_name=('weight',lambda x:x.abs().max()),names=('weight',lambda x:int((x!=0).sum()))).reset_index()
 sector=positions.groupby(['date','sector']).weight.apply(lambda x:x.abs().sum()).groupby(level=0).max()
 return {'annualized_volatility':float(r.std()*np.sqrt(252)),'historical_var_95':var95,'expected_shortfall_95':es95,'max_drawdown':float(dd.min()),'max_gross':float(bydate.gross.max()),'max_net':float(bydate.net.abs().max()),'max_name':float(bydate.max_name.max()),'max_sector':float(sector.max()),'min_active_names':int(bydate.names.min()),'liquidity_proxy':'execution participation and rejection tests','capital':capital,'limitations':['historical VaR is not a loss bound','synthetic returns understate unknown real-world mechanisms','one equity asset class cannot evidence cross-asset contagion']}

def stress_engine(positions,portfolio_daily,executions):
 r=portfolio_daily.net_return.to_numpy();base=float(np.prod(1+r));scenarios={
  'equity_gap_down':float(np.prod(1+(r-.015))),
  'volatility_double':float(np.prod(1+np.clip(r*2,-.20,.20))),
  'correlation_breakdown_equity_sectors':float(np.prod(1+(r-positions.groupby('date').weight.apply(lambda x:x.abs().sum()).reindex(portfolio_daily.date).to_numpy()*.002))),
  'liquidity_cost_x5':float(base-4*executions.total_cost.sum()/1_000_000),
  'parameter_threshold_shift':float(np.prod(1+r*.75))}
 return scenarios

def limit_engine(risk,limits=None):
 limits=limits or {'max_gross':1.0,'max_name':.10,'max_sector':.35,'historical_var_95':.025,'max_drawdown':.15}
 checks=[]
 for k,lim in limits.items():
  val=abs(risk[k]);state='PASS' if val<=lim else ('WARNING' if val<=lim*1.1 else 'BREACH');checks.append({'metric':k,'value':val,'limit':lim,'state':state})
 state='BREACH' if any(x['state']=='BREACH' for x in checks) else ('WARNING' if any(x['state']=='WARNING' for x in checks) else 'PASS')
 return {'overall':state,'checks':checks,'escalation':'HUMAN_RISK_REVIEW' if state!='PASS' else 'NONE'}

def audit_event(seq,actor,action,input_hash,output_hash,policy):
 payload={'sequence':seq,'timestamp_utc':datetime.now(timezone.utc).isoformat(),'actor':actor,'action':action,'input_hash':input_hash,'output_hash':output_hash,'policy':policy}
 payload['event_hash']=sha256_obj(payload);return payload

def run_day5(day4_dir,db_prices,out_dir,corporate_actions_path=None):
 out=Path(out_dir);out.mkdir(parents=True,exist_ok=True);pos=pd.read_csv(Path(day4_dir)/'selected_positions.csv',parse_dates=['date']);daily=pd.read_csv(Path(day4_dir)/'selected_portfolio_daily.csv',parse_dates=['date']);prices=pd.read_csv(db_prices,parse_dates=['date'])
 request={'action':'SIMULATE_EXECUTION','artifact_id':'day4_champion','scope':'synthetic_equities','evidence':['day4_audit_bundle']};suff=semantic_sufficiency_gate(request);rule=action_rule_gate(request)
 ex=execution_simulator(pos,prices);rec=reconcile(ex);risk=independent_risk(pos,daily);stresses=stress_engine(pos,daily,ex);limits=limit_engine(risk)
 adverse={s:reconcile(execution_simulator(pos,prices,s)) for s in ['wide_spread','latency','liquidity_crisis']};actions=pd.read_csv(corporate_actions_path) if corporate_actions_path and Path(corporate_actions_path).exists() else pd.DataFrame();corp=corporate_action_accounting(pos,actions);sample=prices.iloc[0];rng=np.random.default_rng(7);order_types={}
 for typ,kw in [('MARKET',{}),('LIMIT',{'limit_price':float(sample.low)*.99}),('STOP',{'stop_price':float(sample.high)*1.01}),('SCHEDULED',{})]:order_types[typ]=asdict(simulate_order(OrderSpec('DEMO_'+typ,str(pd.Timestamp(sample.date).date()),int(sample.instrument_id),'BUY',10,typ,**kw),sample,rng))
 provenance={'positions_sha256':hashlib.sha256((Path(day4_dir)/'selected_positions.csv').read_bytes()).hexdigest(),'daily_sha256':hashlib.sha256((Path(day4_dir)/'selected_portfolio_daily.csv').read_bytes()).hexdigest()};events=[]
 for i,(a,inp,outp,pol) in enumerate([('semantic_gate',request,suff,suff),('action_gate',request,rule,rule),('execution',request,rec,rule),('risk_review',rec,risk,limits),('governance_acceptance',risk,limits,limits)],1):events.append(audit_event(i,'day5_control_plane',a,sha256_obj(inp),sha256_obj(outp),pol))
 deliberate={'incomplete_request':semantic_sufficiency_gate({'action':'SIMULATE_EXECUTION'}),'live_order':action_rule_gate({'action':'SEND_ORDER'}),'limit_breach':limit_engine({**risk,'max_gross':1.25}),'tampered_provenance':{'decision':'DENY','reason':'hash_mismatch'},'unauthorized_override':action_rule_gate({'action':'OVERRIDE_LIMIT'})}
 decision='PASS_CONTROL_ARCHITECTURE' if suff['allowed'] and rule['decision']=='ALLOW_SIMULATION' and rec['status']=='PASS' and limits['overall']=='PASS' else 'FAIL_CLOSED'
 bundle={'decision':decision,'strategy_promotion':'REMAINS_EXPERIMENTAL','scope':'synthetic equities; simulated execution only','semantic_gate':suff,'action_gate':rule,'reconciliation':rec,'risk_state':risk,'limits':limits,'stress_tests':stresses,'adverse_execution':adverse,'order_type_demonstrations':order_types,'corporate_actions':corp,'provenance':provenance,'audit_events':events,'deliberate_failures':deliberate,'responsibility':{'strategy':'proposes','execution':'simulates','risk':'independently_challenges','governance':'enforces','human':'approves_material_decisions'},'evidence_not_proof':True,'not_applicable':['futures roll: no futures in approved scope','cross-asset contagion: one asset class; equity-sector correlation breakdown used','derivative margin: no derivatives; residual cash accounting retained']}
 ex.to_csv(out/'execution_ledger.csv',index=False);pd.DataFrame([risk]).to_csv(out/'independent_risk_report.csv',index=False);pd.DataFrame(limits['checks']).to_csv(out/'risk_limits.csv',index=False);pd.DataFrame(events).to_json(out/'immutable_audit_events.jsonl',orient='records',lines=True);(out/'day5_audit_bundle.json').write_text(json.dumps(bundle,indent=2));return bundle,ex


## 4. Run the full Day 5 control workflow

`run_day5` consumes the Day 4 handoff and the synthetic market, then executes every control stage in order. It also creates `day5_outputs/` and writes the execution ledger, independent-risk report, limit table, immutable event log, and unified audit bundle. The returned tuple deliberately reports both the control decision and the separate strategy-promotion status: passing controls must never be mistaken for approving the investment strategy.


In [ ]:
bundle,executions=run_day5('day4_outputs','prices_daily.csv','day5_outputs','corporate_actions.csv')
bundle['decision'],bundle['strategy_promotion']

('PASS_CONTROL_ARCHITECTURE', 'REMAINS_EXPERIMENTAL')

## 5. Inspect execution reconciliation

Reconciliation is the first post-trade integrity check. A PASS here means every simulated order has a legal state, order identifiers are unique, and no rejected order carries an impossible fill. Counts of rejected and partial orders reveal whether base liquidity assumptions constrained implementation. This result concerns ledger coherence; it does not prove realistic fills or real-market capacity.


In [ ]:
bundle['reconciliation']

{'status': 'PASS', 'issues': [], 'orders': 436, 'rejected': 0, 'partial': 0}

## 6. Read independent risk and limit states

The risk record combines return-path measures with position-based exposure and concentration measures. The limit record compares selected metrics with declared thresholds and assigns PASS, WARNING, or BREACH. Examine the limitations alongside the numbers: historical VaR is descriptive rather than a loss bound, the data are synthetic, and equity-sector evidence cannot support claims about cross-asset contagion. The independent module may challenge and escalate, but it may not approve its own exception.


In [ ]:
bundle['risk_state'],bundle['limits']

({'annualized_volatility': 0.06037160660786313,
  'historical_var_95': 0.0006883137158916982,
  'expected_shortfall_95': 0.0041310265052890375,
  'max_drawdown': -0.013200966960812965,
  'max_gross': 0.483114779242249,
  'max_net': 0.483114779242249,
  'max_name': 0.0536735596660968,
  'max_sector': 0.13607760269831698,
  'min_active_names': 0,
  'liquidity_proxy': 'execution participation and rejection tests',
  'capital': 1000000,
  'limitations': ['historical VaR is not a loss bound',
   'synthetic returns understate unknown real-world mechanisms',
   'one equity asset class cannot evidence cross-asset contagion']},
 {'overall': 'PASS',
  'checks': [{'metric': 'max_gross',
    'value': 0.483114779242249,
    'limit': 1.0,
    'state': 'PASS'},
   {'metric': 'max_name',
    'value': 0.0536735596660968,
    'limit': 0.1,
    'state': 'PASS'},
   {'metric': 'max_sector',
    'value': 0.13607760269831698,
    'limit': 0.35,
    'state': 'PASS'},
   {'metric': 'historical_var_95',
    'v

## 7. Challenge the candidate and test failure behavior

The first object reports market, volatility, correlation, liquidity-cost, and model-parameter stresses. The second submits intentionally unacceptable conditions to the control plane. The notebook should reject incomplete semantics and live orders, detect a manufactured limit breach and provenance mismatch, and require human approval for an override. These are tests of institutional behavior under adversity. A system that cannot refuse is not governed.


In [ ]:
bundle['stress_tests'],bundle['deliberate_failures']

({'equity_gap_down': 0.11497231518068268,
  'volatility_double': 1.1202219880105997,
  'correlation_breakdown_equity_sectors': 1.0460836422218742,
  'liquidity_cost_x5': 1.0238109432626488,
  'parameter_threshold_shift': 1.044487956390709},
 {'incomplete_request': {'allowed': False,
   'missing': ['artifact_id', 'evidence', 'scope'],
   'reason': 'missing_semantics'},
  'live_order': {'decision': 'DENY',
   'reason': 'research_system_has_no_live_authority'},
  'limit_breach': {'overall': 'BREACH',
   'checks': [{'metric': 'max_gross',
     'value': 1.25,
     'limit': 1.0,
     'state': 'BREACH'},
    {'metric': 'max_name',
     'value': 0.0536735596660968,
     'limit': 0.1,
     'state': 'PASS'},
    {'metric': 'max_sector',
     'value': 0.13607760269831698,
     'limit': 0.35,
     'state': 'PASS'},
    {'metric': 'historical_var_95',
     'value': 0.0006883137158916982,
     'limit': 0.025,
     'state': 'PASS'},
    {'metric': 'max_drawdown',
     'value': 0.013200966960812965,
 

## 8. Package the governed handoff to Day 6

After execution, the `day5_outputs/` directory contains five reusable evidence artifacts:

- `execution_ledger.csv` — order, fill, cost, latency, and status records;
- `independent_risk_report.csv` — the consolidated risk state and its limitations;
- `risk_limits.csv` — metric-by-metric thresholds and control states;
- `immutable_audit_events.jsonl` — a hash-bearing event sequence; and
- `day5_audit_bundle.json` — the complete decision, provenance, responsibility, stress, and failure-test package.

These files are the interface to the next course stage. Day 6 does not need to rediscover what each function means: it can register the validated functions as atomic tools, specify permissions and lifecycle states, compose tools into repeatable skills, and assign those skills to bounded agents. The handoff transfers evidence and capability definitions—not live authority.


## Frequently Asked Questions — NB02  
## Execution, Independent Risk, Governance, and Audit

---

## 1. What is the principal objective of NB02?

NB02 converts the portfolio selected in NB01 from a promising quantitative result into a **governed research object**. NB01 compared prediction models, produced stock-level signals, constructed portfolios, and selected a candidate strategy using out-of-sample evidence. NB02 begins where that process ends.

The notebook asks whether the candidate can pass through simulated execution, order reconciliation, independent risk measurement, limit controls, stress testing, semantic validation, permission controls, provenance verification, and audit recording.

Its purpose is not to authorize trading. It demonstrates the institutional control architecture that must surround a quantitative strategy before promotion, deployment, or live execution could even be considered.

The notebook’s governing principle is:

> **A model may propose an investment decision, but it cannot authorize itself to act.**

---

## 2. What data enter the notebook?

NB02 receives four principal groups of inputs:

1. The selected stock positions produced by NB01.
2. The daily return history of the selected portfolio.
3. The synthetic daily equity-market database.
4. The available corporate-action records.

The principal Day 4 handoff files are:

- `selected_positions.csv`
- `selected_portfolio_daily.csv`

The synthetic market database provides the information required to simulate execution, including:

- Instrument identifiers
- Trading dates
- Opening, high, low, and closing prices
- Adjusted closing prices
- Trading volume
- Estimated bid–ask spreads
- Sector classifications
- Market-regime identifiers

The position file describes the desired portfolio weight for each stock and date. The portfolio-return file supplies the return history used by the independent risk module.

All data remain **synthetic**, and all execution remains **simulated**. Consequently, the notebook provides research and pedagogical evidence rather than proof of live-market profitability, liquidity, or capacity.

---

## 3. How does NB02 connect with NB01?

NB01 produces the candidate portfolio that NB02 evaluates. The relationship between the notebooks can be summarized as:

$
\text{Models}
\rightarrow
\text{Stock Signals}
\rightarrow
\text{Portfolio}
\rightarrow
\text{Execution}
\rightarrow
\text{Risk}
\rightarrow
\text{Governance}
\rightarrow
\text{Audit}
$

NB02 does not repeat the model-selection exercise. It treats the selected positions and portfolio returns as inherited evidence from the preceding stage.

SHA-256 hashes are calculated for the Day 4 position and return files. These hashes identify the exact artifacts subjected to execution and risk review. If an upstream portfolio file were changed, its hash would also change, indicating that the modified portfolio had not passed through the same Day 5 evaluation.

This creates a traceable relationship between the research decision and the control process applied to it.

---

## 4. How are portfolio positions converted into orders?

The notebook compares each stock’s current target weight with its previous target weight:

$
\Delta w_{i,t}=w_{i,t}-w_{i,t-1}
$

The required quantity is estimated using a \$1 million reference portfolio:

$
Q_{i,t}
=
\frac{\Delta w_{i,t}\times \$1{,}000{,}000}{P_{i,t}}
$

where:

- \(Q_{i,t}\) is the quantity to trade;
- \(\Delta w_{i,t}\) is the change in the target weight; and
- \(P_{i,t}\) is the stock’s closing price.

A positive quantity creates a buy order. A negative quantity creates a sell order.

Every order receives a structured record containing:

- An order identifier
- A date
- An instrument identifier
- A buy or sell instruction
- A quantity
- An order type
- Any applicable limit or stop price
- An authority designation

Every generated order is marked:

```python
authority = "SIMULATE_ONLY"
```

This designation is fundamental. The notebook can represent and simulate an order, but it has no authority to transmit that order to a broker or live market.

---

## 5. Which order types are represented?

NB02 demonstrates four order categories:

| Order type | Simplified interpretation |
|---|---|
| Market | Requests execution under the simulated market conditions |
| Limit | Executes only if the specified limit condition is satisfied |
| Stop | Activates only if the simulated stop condition is triggered |
| Scheduled | Represents an order designated for a specified trading session |

The simulator uses daily high and low prices to determine whether simplified limit and stop conditions would have been reached.

These order demonstrations are deliberately bounded. They do not reproduce a complete exchange matching engine, order-book position, routing algorithm, venue selection process, intraday queue, or broker-specific execution protocol.

The scheduled-order example is also a simplified demonstration. A production simulator would need an explicit scheduling model, intraday timestamps, market calendars, auction rules, and venue-specific behavior.

---

## 6. How are execution costs modeled?

The execution simulator separates implementation costs into three components:

### Spread cost

Spread cost represents the estimated cost of crossing part of the bid–ask spread. A buyer generally pays above the reference midpoint, while a seller generally receives less.

### Slippage cost

Slippage represents movement away from the reference price caused by execution conditions. The notebook increases simulated slippage as the trade becomes larger relative to available market volume.

### Market-impact cost

Market impact represents the possibility that the order itself changes the obtainable price. The notebook uses a simplified nonlinear, square-root-style participation proxy.

The simulator also models:

- Latency
- Partial fills
- Liquidity-based rejection
- Wider-spread scenarios
- Liquidity-crisis scenarios
- Participation in estimated dollar volume

The cost calculation is a research approximation. It should not be interpreted as a broker quotation or a prediction of actual implementation shortfall.

In the verified base simulation, the notebook reconciled **436 orders** and estimated aggregate execution costs of:

$
\$8{,}916.92
$

for the \$1 million reference portfolio.

---

## 7. What does execution reconciliation mean?

Execution reconciliation determines whether the simulated order and fill records are internally coherent.

The notebook verifies that:

- Every execution has a recognized status.
- Order identifiers are unique.
- Rejected orders have zero filled quantity.
- Orders and fills can be joined correctly.
- Partial fills are represented consistently.
- No impossible combination appears in the ledger.

The permitted execution states are:

```python
{"FILLED", "PARTIAL", "REJECTED"}
```

The verified base simulation produced:

```python
{
    "status": "PASS",
    "issues": [],
    "orders": 436,
    "rejected": 0,
    "partial": 0
}
```

This PASS result means that the simulated ledger is internally consistent. It does not prove that the fills are realistic, that real-market liquidity would be sufficient, or that the portfolio possesses live trading capacity.

---

## 8. How are corporate actions treated?

The notebook recognizes dividends and stock splits within the approved synthetic-equity scope.

Prices are assumed to be adjusted. Dividend cash is reported separately so that its economic effect is not counted twice. Split events are identified and reported as part of the accounting record.

The notebook explicitly classifies the following controls as not applicable:

- Futures-roll accounting
- Derivative-margin modeling
- Full cross-asset contagion analysis

These items are not applicable because the approved universe contains synthetic equities rather than futures, options, or multiple asset classes.

A control that is outside the experiment’s scope must be reported as **not applicable**. It should never be silently represented as having passed.

---

## 9. Why is risk measurement described as independent?

The risk module is separated conceptually from the strategy-selection process. The component that proposes a portfolio should not have exclusive responsibility for deciding whether its own risks are acceptable.

The independent risk module measures:

- Annualized volatility
- Historical value at risk
- Expected shortfall
- Maximum drawdown
- Gross exposure
- Net exposure
- Maximum single-name exposure
- Maximum sector exposure
- Minimum number of active positions
- Execution-based liquidity indicators

This separation creates an institutional challenge function:

> **Strategy proposes; independent risk evaluates and challenges.**

The risk module can identify a breach and require escalation. It cannot approve its own exception, promote the strategy, or authorize live execution.

---

## 10. What were the principal independent-risk results?

The verified base-case results were:

| Risk measure | Result |
|---|---:|
| Annualized volatility | 6.04% |
| Historical VaR at 95% | 0.069% |
| Expected shortfall at 95% | 0.413% |
| Maximum drawdown | 1.32% |
| Maximum gross exposure | 48.31% |
| Maximum single-name exposure | 5.37% |
| Maximum sector exposure | 13.61% |

All declared base thresholds received a `PASS` status.

These results must remain attached to their limitations. Historical VaR is not a maximum-loss estimate. Expected shortfall depends on the observed or simulated distribution. Synthetic returns cannot include every real-market mechanism. Furthermore, evidence from one equity universe cannot demonstrate resilience to genuine cross-asset contagion.

A favorable risk table is evidence about the simulated base case, not proof that the portfolio is safe.

---

## 11. How does the limit engine operate?

The limit engine compares selected risk measurements with predetermined thresholds.

The notebook monitors:

| Metric | Limit |
|---|---:|
| Maximum gross exposure | 100% |
| Maximum single-name exposure | 10% |
| Maximum sector exposure | 35% |
| Historical VaR at 95% | 2.5% |
| Maximum drawdown | 15% |

Each measure receives one of three states:

- `PASS`
- `WARNING`
- `BREACH`

A value within its limit receives `PASS`. A value exceeding the limit by no more than 10% receives `WARNING`. A more substantial violation receives `BREACH`.

The overall state becomes:

- `BREACH` if any individual control is in breach;
- `WARNING` if no control is breached but at least one generates a warning; or
- `PASS` if all controls pass.

A warning or breach produces the escalation instruction:

```python
"HUMAN_RISK_REVIEW"
```

A limit violation is therefore not merely displayed. It changes the governance state of the research process.

---

## 12. How is the portfolio stressed?

The notebook applies several adverse scenarios:

- A persistent equity gap-down shock
- A doubling of portfolio volatility
- A breakdown in normal equity-sector diversification
- A fivefold increase in modeled liquidity costs
- A reduction in model or parameter effectiveness
- Wider market spreads
- Greater simulated latency
- A simulated liquidity crisis

The most important warning comes from the equity gap-down scenario. Under that path, the terminal wealth factor falls to approximately:

$
0.115
$

This represents a catastrophic deterioration relative to the base path.

The stress engine is not intended to predict the exact next crisis. Its purpose is to reveal how the strategy behaves when important assumptions are weakened or violated.

Stress testing therefore asks:

> **What could happen if the environment is materially worse than the one used to select the strategy?**

---

## 13. What is the semantic-sufficiency gate?

The **semantic-sufficiency gate** determines whether a request contains enough information to be understood and evaluated safely.

It does not decide whether the action is authorized. Its purpose is to prevent the system from acting on an incomplete, ambiguous, or undocumented instruction.

Every governed request must contain four fields:

| Required field | Meaning | Example in NB02 |
|---|---|---|
| `action` | The operation being requested | `SIMULATE_EXECUTION` |
| `artifact_id` | The model, portfolio, or governed object affected | `day4_champion` |
| `scope` | The assets and operational boundary | `synthetic_equities` |
| `evidence` | The artifacts supporting the request | `day4_audit_bundle` |

The principal NB02 request is:

```python
request = {
    "action": "SIMULATE_EXECUTION",
    "artifact_id": "day4_champion",
    "scope": "synthetic_equities",
    "evidence": ["day4_audit_bundle"]
}
```

The semantic gate compares the supplied fields with the required set:

```python
required = {"action", "artifact_id", "scope", "evidence"}
missing = sorted(required - set(request))
```

If nothing is missing, the gate returns:

```python
{
    "allowed": True,
    "missing": [],
    "reason": "complete"
}
```

If required fields are absent, the gate returns:

```python
{
    "allowed": False,
    "missing": ["artifact_id", "evidence", "scope"],
    "reason": "missing_semantics"
}
```

The deliberate-failure test submits only:

```python
{"action": "SIMULATE_EXECUTION"}
```

That request is refused because the system does not know:

- Which artifact should be processed
- The scope within which the action is permitted
- The evidence supporting the request

The semantic gate is designed to **fail closed**. Missing information does not produce a default approval.

A semantically complete request allows the system to answer four preliminary questions:

1. What action is being requested?
2. Which governed artifact would be affected?
3. Within what operational boundary should the action occur?
4. What evidence supports the request?

Semantic completeness does not imply authorization. It means only that the system understands the request sufficiently to evaluate it.

---

## 14. What is the action-rule gate?

The **action-rule gate** determines whether the requested action is permitted under the notebook’s authority model.

The implemented actions fall into three governance categories:

| Action category | Examples | Decision |
|---|---|---|
| Bounded research and simulation | `SIMULATE_EXECUTION` | `ALLOW_SIMULATION` |
| Prohibited live or production action | `LIVE_EXECUTION`, `SEND_ORDER`, `ALTER_PRODUCTION` | `DENY` |
| Material decision | `OVERRIDE_LIMIT`, `PROMOTE_STRATEGY` | `REQUIRE_HUMAN_APPROVAL` |

The notebook implements the following decision logic:

```python
def action_rule_gate(request):
    if request.get("action") in {
        "LIVE_EXECUTION",
        "SEND_ORDER",
        "ALTER_PRODUCTION"
    }:
        return {
            "decision": "DENY",
            "reason": "research_system_has_no_live_authority"
        }

    if request.get("action") in {
        "OVERRIDE_LIMIT",
        "PROMOTE_STRATEGY"
    }:
        return {
            "decision": "REQUIRE_HUMAN_APPROVAL",
            "reason": "material_decision"
        }

    return {
        "decision": "ALLOW_SIMULATION",
        "reason": "bounded_research_action"
    }
```

### `ALLOW_SIMULATION`

The gate permits an action that remains inside the approved research boundary.

For the principal NB02 request, the result is:

```python
{
    "decision": "ALLOW_SIMULATION",
    "reason": "bounded_research_action"
}
```

This decision permits only simulated execution using synthetic data. It does not authorize live trading, deployment, broker connectivity, strategy promotion, or expansion of the declared scope.

### `DENY`

The actions `LIVE_EXECUTION`, `SEND_ORDER`, and `ALTER_PRODUCTION` are categorically denied.

For example:

```python
action_rule_gate({"action": "SEND_ORDER"})
```

returns:

```python
{
    "decision": "DENY",
    "reason": "research_system_has_no_live_authority"
}
```

A successful backtest, reconciliation result, or risk report cannot create authority that the system was never granted.

### `REQUIRE_HUMAN_APPROVAL`

The actions `OVERRIDE_LIMIT` and `PROMOTE_STRATEGY` are treated as material decisions.

For example:

```python
action_rule_gate({"action": "OVERRIDE_LIMIT"})
```

returns:

```python
{
    "decision": "REQUIRE_HUMAN_APPROVAL",
    "reason": "material_decision"
}
```

This response does not approve the override. It stops automatic processing and transfers the decision to an authorized human governance owner.

The same principle applies to strategy promotion. A model or agent may assemble evidence and recommend promotion, but it cannot approve its own promotion.

### What is the limitation of the current action gate?

The gate is a transparent educational implementation rather than a production authorization system. Actions not included in the prohibited or material-decision sets fall through to `ALLOW_SIMULATION`.

A production implementation should use an explicit allowlist and deny every unknown action by default. It should also validate authenticated identities, roles, environments, policy versions, scopes, budgets, and artifact states.

A stronger production rule would follow this structure:

```python
if action in explicitly_permitted_research_actions:
    return "ALLOW_SIMULATION"
elif action in material_actions:
    return "REQUIRE_HUMAN_APPROVAL"
else:
    return "DENY"
```

This would prevent an unrecognized action from receiving permission merely because it was not included in the original denial list.

---

## 15. How do the semantic and action gates work together?

The two gates answer different questions and must operate in sequence:

$
\text{Request}
\rightarrow
\text{Semantic Gate}
\rightarrow
\text{Action Gate}
\rightarrow
\text{Bounded Action, Denial, or Escalation}
$

The semantic gate asks:

> **Is the request complete enough to understand?**

The action gate then asks:

> **Is the understood action permitted?**

The possible outcomes are:

| Semantic result | Action result | Operational outcome |
|---|---|---|
| Incomplete | Not evaluated further | Stop and request missing semantics |
| Complete | `ALLOW_SIMULATION` | Proceed within the declared research scope |
| Complete | `DENY` | Stop and record the denial |
| Complete | `REQUIRE_HUMAN_APPROVAL` | Pause and escalate to an authorized human |

This separation prevents two important failures.

First, the system cannot authorize an action whose meaning is incomplete. Second, a clearly expressed request cannot cross a permission boundary merely because it is semantically complete.

In the principal NB02 experiment, the request is complete and asks only for simulated execution. It therefore passes both gates and proceeds to order construction, simulated fills, reconciliation, independent risk, stress testing, and audit.

---

## 16. Which deliberate governance failures are tested?

NB02 intentionally submits unacceptable conditions to the control architecture.

### Incomplete request

A request containing only an action fails the semantic gate because it lacks an artifact identifier, scope, and supporting evidence.

### Live-order request

A request to send a live order receives:

```python
{
    "decision": "DENY",
    "reason": "research_system_has_no_live_authority"
}
```

### Manufactured risk breach

The notebook artificially increases maximum gross exposure to 125%. This exceeds the 100% limit and produces:

```python
{
    "overall": "BREACH",
    "escalation": "HUMAN_RISK_REVIEW"
}
```

### Provenance mismatch

A simulated hash mismatch receives:

```python
{
    "decision": "DENY",
    "reason": "hash_mismatch"
}
```

### Unauthorized override

A request to override a limit receives:

```python
{
    "decision": "REQUIRE_HUMAN_APPROVAL",
    "reason": "material_decision"
}
```

These tests demonstrate that a governed system must be capable of refusing, stopping, or escalating an action.

> **A system that cannot refuse is not governed.**

---

## 17. How are governance decisions incorporated into the audit record?

The outputs of the semantic gate, action gate, execution simulator, independent risk review, and governance assessment become part of the Day 5 audit evidence.

Each audit event contains:

- A sequence number
- A UTC timestamp
- The responsible actor
- The action performed
- A hash of the input
- A hash of the output
- The applicable policy record
- A hash of the event itself

This makes it possible to review:

- What the system was asked to do
- Which artifact was affected
- Which evidence supported the request
- Which scope was declared
- Which permission rule was applied
- Whether the action was allowed, denied, or escalated
- Which outputs were produced

The notebook writes these records to:

```text
immutable_audit_events.jsonl
```

The events are hash-bearing, but the educational implementation does not by itself create a fully immutable production audit system. Production-grade immutability would require protected append-only storage, access controls, policy versioning, digital signatures or chained hashes, secure timestamps, independent monitoring, and tested retention procedures.

The notebook demonstrates the architecture and evidence model on which those stronger controls can be built.

---

## 18. How is the success of NB02 measured?

NB02 evaluates success across several independent dimensions:

1. **Execution integrity:** Orders and fills must reconcile.
2. **Cost transparency:** Spread, slippage, and impact must be attributed.
3. **Risk control:** Monitored measures must be compared with declared limits.
4. **Stress behavior:** Material vulnerabilities must remain visible.
5. **Semantic sufficiency:** Incomplete requests must fail closed.
6. **Permission control:** Unauthorized actions must be denied.
7. **Human escalation:** Material decisions must not be self-approved.
8. **Provenance:** Inherited evidence must be cryptographically identifiable.
9. **Auditability:** Material decisions must create reviewable records.
10. **Scope discipline:** Synthetic evidence must not be presented as live proof.

The notebook’s final control decision is:

```text
PASS_CONTROL_ARCHITECTURE
```

The strategy’s status remains:

```text
REMAINS_EXPERIMENTAL
```

Live execution remains:

```text
DENIED
```

These conclusions are intentionally separate. The control architecture behaved as designed, but the investment strategy was not approved for deployment.

---

## 19. What audit and handoff artifacts does the notebook produce?

NB02 produces five principal artifacts:

| Artifact | Purpose |
|---|---|
| `execution_ledger.csv` | Records orders, fills, costs, latency, status, and execution reasons |
| `independent_risk_report.csv` | Preserves the consolidated independent-risk state |
| `risk_limits.csv` | Records monitored metrics, thresholds, and control states |
| `immutable_audit_events.jsonl` | Stores the hash-bearing sequence of material events |
| `day5_audit_bundle.json` | Consolidates decisions, provenance, stresses, failures, limitations, and responsibilities |

Together, these files form the notebook’s **output contract**.

The next course stage can consume these artifacts without having to reconstruct the entire Day 5 analysis. The handoff transfers evidence, validated functions, declared limitations, and authority boundaries.

It does not transfer permission to trade.

---

## 20. How does NB02 connect with the next stages of the course?

NB02 supplies the control architecture required by Day 6.

The execution simulator can become a governed execution-simulation tool. The risk engine can support an independent-risk skill. The semantic and action gates can become reusable governance controls. The audit events and provenance hashes can support an audit-verification capability.

The subsequent progression is:

| Stage | Institutional capability |
|---|---|
| Day 6 | Atomic tools, repeatable skills, permissions, registries, and bounded agents |
| Day 7 | Research, Strategy, Portfolio, and Risk-and-Governance constellations |
| Day 8 | Governed meta-agent planning and delegation |
| Day 9 | Integration into a fail-closed autonomous research system |
| Day 10 | Research-to-LEAN bridge, experiment registry, and system validation |

Greater architectural sophistication must not create additional authority automatically. Tools, skills, agents, constellations, and the meta-agent must inherit the limitations established in NB02.

The progression is therefore:

$
\text{Validated Functions}
\rightarrow
\text{Governed Tools}
\rightarrow
\text{Repeatable Skills}
\rightarrow
\text{Bounded Agents}
\rightarrow
\text{Agent Constellations}
\rightarrow
\text{Meta-Agent}
\rightarrow
\text{Autonomous Research System}
$

---

## 21. What is the final lesson the user should retain?

A successful prediction model is only the beginning of an institutional investment process.

Forecasts must become stock-level signals. Signals must become constrained portfolios. Portfolio changes must become explicit orders. Execution assumptions and costs must be measured. Orders and fills must reconcile. Independent risk must challenge the proposal. Stress tests must expose vulnerabilities. Semantic gates must ensure that requests are sufficiently complete. Action gates must enforce authority boundaries. Governance must escalate material decisions. Audit must preserve the evidence.

The central doctrine of NB02 is:

> **The model proposes. The semantic gate verifies meaning. The action gate verifies authority. Execution simulates. Independent risk challenges. Governance enforces. Audit preserves the evidence. The human remains accountable.**

NB02 does not prove that the strategy is profitable, robust, scalable, or ready for live markets. It demonstrates how quantitative intelligence can be transformed into a bounded, inspectable, reproducible, and governable institutional capability.

## Conclusion: main results and connection to the next stages

NB02 has converted the Day 4 champion from a performance claim into a governed research object. On the one-million-dollar reference portfolio, changes in selected target weights generated 436 simulated orders. All orders reconciled: there were no duplicate identifiers, illegal states, rejected orders with fills, base-case rejections, or partial fills. Modeled spread, slippage, and market-impact costs totaled $8,916.92. This is an implementation estimate inside the synthetic market, not proof of obtainable broker fills or real capacity. Its value is that trading friction is now explicit, attributed, and auditable instead of being hidden inside a frictionless backtest.

The independent risk review also passed its declared base limits. Annualized volatility was 6.04%, historical 95% VaR was 0.069%, expected shortfall was 0.413%, and maximum drawdown was 1.32%. Maximum gross exposure was 48.31%, maximum single-name exposure 5.37%, and maximum sector exposure 13.61%. Each monitored threshold received a PASS and no base-case escalation was required. These favorable numbers must remain attached to their limitations: historical VaR is not a worst-case bound, synthetic returns cannot contain every market mechanism, and one equity universe cannot demonstrate cross-asset resilience.

Stress testing supplies the notebook’s most important warning. The equity gap-down path reduces terminal wealth to approximately 0.115, a catastrophic outcome that dominates the comfortable base statistics. Other scenarios perturb volatility, sector correlation, liquidity cost, and parameter effectiveness. Meanwhile, deliberate failures behave correctly: incomplete requests are rejected, live orders are denied, a forced gross-exposure breach escalates to human risk review, a provenance mismatch fails, and an unauthorized override requires human approval. The final verdict is therefore precise: `PASS_CONTROL_ARCHITECTURE`, while `REMAINS_EXPERIMENTAL` is retained for the strategy. The notebook validates the control machinery, not the investment thesis and never live trading.

To concatenate NB02 with Day 6, preserve `day5_outputs/` as the formal input contract. The execution ledger becomes evidence for execution-simulation tools; the risk and limit files support an independent-risk skill; the JSONL events support audit verification; and the unified audit bundle carries scope, hashes, decisions, limitations, failure tests, and responsibility. Day 6 can wrap the proven functions in atomic tool specifications, register permissions and lifecycle states, compose repeatable skills, and assign bounded agents without expanding authority. Day 7 can then coordinate those agents into Research, Strategy, Portfolio, and Risk-and-Governance constellations. Day 8 adds a meta-agent that plans and delegates across validated registries. Day 9 integrates the pieces into a fail-closed autonomous research runtime, and Day 10 constructs the governed research-to-LEAN bridge and experiment registry.

The final takeaway is institutional rather than merely statistical: a model proposes; execution makes assumptions visible; independent risk challenges; governance constrains; audit preserves evidence; and a human remains accountable for material decisions. That chain is the course’s mechanism for turning quantitative intelligence into bounded, inspectable autonomy. Only after those layers survive matched real data, external validation, platform reconciliation, security review, and explicit human authorization could the research artifact even become a candidate for deployment or trading.
